# Insurance Policy RAG on Colab

Runs the `policy_rag` package from this repository on a free Colab GPU: ask your own questions, or run the evaluation.

1. *Runtime → Change runtime type → T4 GPU*, then run the cells in order.
2. You need the policy text: the my:Optima Secure PDF (cell 2 turns it into a chunk file in seconds), or a chunk file
   you made before. Neither is stored in the repository (copyright HDFC ERGO; see `data/README.md`).
3. First answer: about 5–8 minutes of model downloads (about 20 GB). With thinking on, answers take about 47 s each
   (median of the published run; the slowest 5% over 2 minutes).

In [ ]:
# 1) Get the code and install it (Colab already has PyTorch)
import os, torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")
if not os.path.exists("/content/insurance-policy-rag"):
    !git clone -q https://github.com/RamkrushnaSaner/insurance-policy-rag.git /content/insurance-policy-rag
%cd /content/insurance-policy-rag
!pip install -q -e ".[models,ingest]"
import sys
sys.path.insert(0, "/content/insurance-policy-rag/src")   # the editable install is seen only after a restart


In [ ]:
import sys
# 2) The policy: upload the PDF (it is chunked here) or an existing chunk file (.json)
from pathlib import Path
from google.colab import files
Path("data").mkdir(exist_ok=True)
uploaded = files.upload()
name = next(iter(uploaded))
if name.lower().endswith(".pdf"):
    Path("data/policy.pdf").write_bytes(uploaded[name])
    !{sys.executable} -m policy_rag chunk-pdf data/policy.pdf data/policy_chunks.json
else:
    Path("data/policy_chunks.json").write_bytes(uploaded[name])
!{sys.executable} -m policy_rag info


## Ask questions
`ask()` prints the passages the answer model read (best first) and the answer. Every answer is cached in
`runs/answer_cache.jsonl`: asking the same question again costs nothing.

In [ ]:
# 3) Build the system once (embedder, reranker and indexes; the answer model loads on the first answer)
from policy_rag import RAGSystem
rag = RAGSystem.from_config("configs/default.yaml")


In [ ]:
r = rag.ask("What is the waiting period for pre-existing diseases?")


## Follow-up questions (conversation memory)
Pass a conversation to `ask()` and follow-ups like *"does it apply after an accident?"* work: the system rewrites the
follow-up into a complete question from the last 3 turns (shown as *searched as*), then searches and answers it as
usual. The answer still comes only from the policy text. A follow-up costs one extra model call. `rag.chat()` opens an
interactive chat (type `/help` for commands).

In [ ]:
chat = rag.new_conversation()
rag.ask("What is the waiting period for kidney stones?", conversation=chat)
r = rag.ask("Does it apply if the problem is caused by an accident?", conversation=chat)


In [ ]:
# Interactive chat: type questions; /new, /save, /history, /quit
# rag.chat()


## Evaluate
The test set has 113 questions (25 dev, 88 held-out test). Choose what to run:

| `SPLITS` | answers | time with thinking on |
|---|---|---|
| `["dev"]` | 25 | about 25 minutes |
| `["dev", "test"]` | 113 | about 2 hours |

Answers in the cache are not generated again. If you have the `answer_cache.jsonl` (or results zip) of an earlier run
with the same settings, set `RESUME = True` and upload it: the run then only re-checks those answers.
Download the results zip at the end; it contains policy text, so keep it out of public repositories.

In [ ]:
# 4) Evaluation run (reuses the system built above, so the models are not loaded twice)
from policy_rag.evaluation.runner import Evaluation
SPLITS = ["dev"]
RESUME = False
resume = [next(iter(files.upload()))] if RESUME else []
run = Evaluation(None, "baseline", mode="full", splits=SPLITS, resume_from=resume, system=rag).run(make_zip_file=True)
run["tables"]["scorecard"]


In [ ]:
files.download(run["zip"])


## Evaluate the conversation memory
23 short conversations (26 follow-ups) from `testsets/optima_secure_followups.json`. Every follow-up is answered three
ways: as typed without memory, rewritten by memory, and as the complete question. Turn-1 and complete-question answers
come from the cache when they were generated before (upload the baseline's `run.json` or results zip with
`RESUME = True` above). About 45–60 minutes with thinking on; the decision rule is in the report.

In [ ]:
from policy_rag.evaluation.dialogues import DialogueEvaluation
dlg = DialogueEvaluation(None, "memory", system=rag, resume_from=resume).run(make_zip_file=True)
print(dlg["decision"])
files.download(dlg["zip"])


## Experiments
Each experiment is a small config file that changes one thing (`configs/experiments/`), or a `--set` override.
Run it under its own name, then compare it with the baseline on the held-out test split:

```
!python -m policy_rag evaluate --config configs/experiments/no_thinking.yaml --name no_thinking --splits dev test
!python -m policy_rag compare runs/baseline/run.json runs/no_thinking/run.json --splits test
```
Restart the runtime first (*Runtime → Restart session*) if the models from cell 3 still fill the GPU.